# Attention Mechanisms: Deep Dive
### Interactive Notebook for AI/ML Interview Preparation

📺 **Video Lecture:** [https://youtu.be/te7D9Al7mpw](https://youtu.be/te7D9Al7mpw)

## Imports & Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
sns.set_style('whitegrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('All libraries loaded successfully!')

---
## 1. Scaled Dot-Product Attention

The foundation of modern attention mechanisms. Computes:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

where:
- **Q (Query)**: What are we looking for?
- **K (Key)**: What information is available?
- **V (Value)**: What information do we extract?
- **Scaling factor** ($\sqrt{d_k}$): Prevents softmax saturation for large dimensions

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    Args:
        Q: (batch, seq_len, d_k)
        K: (batch, seq_len, d_k)
        V: (batch, seq_len, d_v)
        mask: optional mask to prevent attention to certain positions
    Returns:
        output: (batch, seq_len, d_v)
        attention_weights: (batch, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute similarity scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
    print(f'Scores shape: {scores.shape}')
    print(f'Scores (before softmax):\n{scores}')
    
    # Step 2: Apply mask (if provided)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 3: Apply softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)
    print(f'Attention weights (after softmax):\n{attention_weights}')
    
    # Step 4: Multiply by values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Example: Single query attending to 5 keys
batch_size = 1
seq_len = 5
d_k = 4  # Dimension of keys/queries
d_v = 3  # Dimension of values

Q = torch.randn(batch_size, 1, d_k)  # 1 query
K = torch.randn(batch_size, seq_len, d_k)  # 5 keys
V = torch.randn(batch_size, seq_len, d_v)  # 5 values

print('=== Scaled Dot-Product Attention Example ===')
output, weights = scaled_dot_product_attention(Q, K, V)
print(f'\nOutput shape: {output.shape}')
print(f'Output:\n{output}')

---
## 2. Additive (Bahdanau) vs Multiplicative (Luong) Attention

**Additive Attention (Bahdanau):**
$$\text{score}(s, h_i) = v^T \tanh(W_1 s + W_2 h_i)$$
- More flexible, can handle different dimensions
- Requires learning additional parameters

**Multiplicative Attention (Luong):**
$$\text{score}(s, h_i) = s^T W h_i$$
- Faster (single matrix multiplication)
- Used in Transformers (scaled dot-product is a special case)

In [ ]:
class BahdanauAttention(nn.Module):
    """Additive (Bahdanau) Attention"""
    def __init__(self, query_dim, key_dim, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(query_dim, hidden_dim)
        self.W2 = nn.Linear(key_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1)
    
    def forward(self, query, keys):
        """
        query: (batch, query_dim)
        keys: (batch, seq_len, key_dim)
        returns: attention_weights (batch, seq_len)
        """
        # Expand query for broadcasting
        query = query.unsqueeze(1)  # (batch, 1, query_dim)
        
        # score = v^T * tanh(W1*query + W2*keys)
        scores = self.v(torch.tanh(self.W1(query) + self.W2(keys))).squeeze(-1)
        weights = F.softmax(scores, dim=-1)
        return weights

class LuongAttention(nn.Module):
    """Multiplicative (Luong) Attention"""
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Linear(dim, dim)
    
    def forward(self, query, keys):
        """
        query: (batch, dim)
        keys: (batch, seq_len, dim)
        returns: attention_weights (batch, seq_len)
        """
        # score = query^T * W * keys^T
        query = query.unsqueeze(1)  # (batch, 1, dim)
        scores = torch.matmul(query, self.W(keys).transpose(-2, -1)).squeeze(1)
        weights = F.softmax(scores, dim=-1)
        return weights

# Comparison
batch_size = 2
seq_len = 6
dim = 8

query = torch.randn(batch_size, dim)
keys = torch.randn(batch_size, seq_len, dim)

bahdanau = BahdanauAttention(dim, dim, hidden_dim=16)
luong = LuongAttention(dim)

with torch.no_grad():
    bahdanau_weights = bahdanau(query, keys)
    luong_weights = luong(query, keys)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bahdanau
sns.heatmap(bahdanau_weights.detach().numpy(), annot=True, fmt='.3f', 
            cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title('Bahdanau (Additive) Attention')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Batch')

# Luong
sns.heatmap(luong_weights.detach().numpy(), annot=True, fmt='.3f', 
            cmap='Greens', ax=axes[1], cbar=True)
axes[1].set_title('Luong (Multiplicative) Attention')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Batch')

plt.tight_layout()
plt.show()

print('Bahdanau weights shape:', bahdanau_weights.shape)
print('Luong weights shape:', luong_weights.shape)

---
## 3. Multi-Head Attention

Instead of single attention, apply attention multiple times in parallel:
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**Benefits:**
- Each head attends to different parts of the input
- Captures diverse types of relationships
- Parallel computation improves efficiency

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention from scratch"""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None):
        batch_size = Q.shape[0]
        
        # Linear projections
        Q = self.W_q(Q)  # (batch, seq_len, d_model)
        K = self.W_k(K)
        V = self.W_v(V)
        
        # Reshape for multi-head: (batch, seq_len, d_model) -> (batch, seq_len, num_heads, d_k)
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        # Shape: (batch, num_heads, seq_len, d_k)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        context = torch.matmul(attention_weights, V)  # (batch, num_heads, seq_len, d_k)
        
        # Concatenate heads
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, -1, self.d_model)
        
        # Final output projection
        output = self.W_o(context)
        
        return output, attention_weights

# Test custom multi-head attention
batch_size = 2
seq_len = 4
d_model = 8
num_heads = 2

Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

custom_mha = MultiHeadAttention(d_model, num_heads)
with torch.no_grad():
    custom_output, custom_weights = custom_mha(Q, K, V)

print('Custom Multi-Head Attention:')
print(f'Output shape: {custom_output.shape}')
print(f'Attention weights shape: {custom_weights.shape} (batch, heads, seq_len, seq_len)')

# Compare with PyTorch's built-in
torch_mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
with torch.no_grad():
    torch_output, torch_weights = torch_mha(Q, K, V)

print(f'\nPyTorch MultiheadAttention:')
print(f'Output shape: {torch_output.shape}')
print(f'Weights shape: {torch_weights.shape}')

---
## 4. Self-Attention vs Cross-Attention

**Self-Attention:**
- Q, K, V all come from the same sequence
- Tokens attend to all other tokens in the same sequence
- Used in: Encoder, encoder self-attention layers

**Cross-Attention:**
- Q comes from one sequence, K and V from another
- Decoder attends to encoder outputs
- Enables alignment between source and target

In [ ]:
class TransformerBlock(nn.Module):
    """Simple Transformer block with self-attention and cross-attention"""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True, dropout=dropout)
        self.cross_attention = nn.MultiheadAttention(d_model, num_heads, batch_first=True, dropout=dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, tgt, memory=None):
        # Self-attention
        tgt_attn, _ = self.self_attention(tgt, tgt, tgt)
        tgt = self.norm1(tgt + self.dropout(tgt_attn))
        
        # Cross-attention (if memory is provided)
        if memory is not None:
            cross_attn, _ = self.cross_attention(tgt, memory, memory)
            tgt = self.norm2(tgt + self.dropout(cross_attn))
        
        # Feed-forward
        ff = self.feed_forward(tgt)
        tgt = self.norm3(tgt + self.dropout(ff))
        
        return tgt

# Example: Encoder-Decoder setup
d_model = 64
num_heads = 4
src_seq_len = 5
tgt_seq_len = 4

# Simulated encoder output (memory)
memory = torch.randn(2, src_seq_len, d_model)
# Decoder target sequence
target = torch.randn(2, tgt_seq_len, d_model)

decoder_layer = TransformerBlock(d_model, num_heads)

with torch.no_grad():
    output = decoder_layer(target, memory=memory)

print('Encoder-Decoder with Self + Cross Attention:')
print(f'Memory (encoder) shape: {memory.shape}')
print(f'Target (decoder) shape: {target.shape}')
print(f'Decoder output shape: {output.shape}')
print('\nFlow:')
print('  1. Target -> Self-Attention (attend to itself)')
print('  2. Result -> Cross-Attention (attend to encoder outputs)')
print('  3. Result -> Feed-Forward')

---
## 5. KV-Cache for Efficient Inference

During autoregressive generation, we recompute K and V for all previous tokens at each step.
**KV-Cache** stores them to avoid recomputation.

**Impact:**
- Reduces memory bandwidth (don't reload K,V repeatedly)
- Reduces computation: O(n²) → O(n) for generation
- For a 100-token sequence: ~100x faster with caching

In [ ]:
class AutoregressiveAttentionWithKVCache(nn.Module):
    """Attention with KV-Cache for efficient generation"""
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, kv_cache=None):
        """
        x: (batch, seq_len, d_model) - during prefill, entire input; during decode, only new token
        kv_cache: dict with 'k' and 'v' tensors from previous steps
        returns: output, updated_cache
        """
        batch_size = x.shape[0]
        
        Q = self.W_q(x)  # (batch, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Reshape for multi-head
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Concatenate with cached K,V
        if kv_cache is not None:
            K = torch.cat([kv_cache['k'], K], dim=-2)  # (batch, heads, past_len + new_len, d_k)
            V = torch.cat([kv_cache['v'], V], dim=-2)
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)
        
        # Output
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, -1, self.d_model)
        output = self.W_o(context)
        
        # Return updated cache
        new_cache = {'k': K, 'v': V}
        return output, new_cache

# Simulation: Generate 10 tokens with and without cache
d_model = 64
num_heads = 4
seq_len = 10

attention_layer = AutoregressiveAttentionWithKVCache(d_model, num_heads)

# WITHOUT KV-Cache: recompute at each step
computation_no_cache = []
for step in range(1, seq_len + 1):
    # Each step processes entire sequence up to current position
    # K,V computation: O(step * d_model)
    # Attention: O(step²)
    computation_no_cache.append(step ** 2)

# WITH KV-Cache: only new token contributes
computation_with_cache = [step for step in range(1, seq_len + 1)]  # Only new K,V computed

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

steps = list(range(1, seq_len + 1))
ax1.plot(steps, computation_no_cache, 'ro-', label='Without KV-Cache', linewidth=2, markersize=8)
ax1.plot(steps, computation_with_cache, 'go-', label='With KV-Cache', linewidth=2, markersize=8)
ax1.set_xlabel('Generation Step', fontsize=12)
ax1.set_ylabel('Attention Computation (operations)', fontsize=12)
ax1.set_title('KV-Cache Reduces Computation', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Cumulative
cumsum_no_cache = np.cumsum(computation_no_cache)
cumsum_with_cache = np.cumsum(computation_with_cache)
ax2.plot(steps, cumsum_no_cache, 'ro-', label='Without KV-Cache', linewidth=2, markersize=8)
ax2.plot(steps, cumsum_with_cache, 'go-', label='With KV-Cache', linewidth=2, markersize=8)
ax2.set_xlabel('Generation Step', fontsize=12)
ax2.set_ylabel('Cumulative Operations', fontsize=12)
ax2.set_title('Cumulative Computation Savings', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Speedup at step {seq_len}: {cumsum_no_cache[-1] / cumsum_with_cache[-1]:.1f}x faster with KV-Cache')

---
## 6. Attention Visualization with BERT

Extract real attention weights from a pre-trained BERT model and visualize which tokens attend to which.

In [ ]:
# Load BERT and extract attention weights
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_attentions=True)
model.eval()

# Example sentence
sentence = "The cat sat on the mat and was very happy."
inputs = tokenizer.encode(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

# Forward pass with attention
with torch.no_grad():
    outputs = model(inputs)
    attention = outputs[-1]  # Attention weights from all layers

print(f'Sentence: {sentence}')
print(f'Tokens: {tokens}')
print(f'Number of layers: {len(attention)}')
print(f'Attention shape per layer: {attention[0].shape}')  # (batch=1, heads=12, seq_len, seq_len)

# Visualize attention from last layer, first head
last_layer_attention = attention[-1][0]  # (12 heads, seq_len, seq_len)
first_head = last_layer_attention[0].cpu().numpy()  # First attention head

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Head 1
sns.heatmap(first_head, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0], 
            xticklabels=tokens, yticklabels=tokens, cbar_kws={'label': 'Attention Weight'})
axes[0].set_title('BERT - Last Layer, Head 1', fontsize=13)
axes[0].set_xlabel('Attending to (Keys)', fontsize=11)
axes[0].set_ylabel('Attending from (Queries)', fontsize=11)

# Head 2 for comparison
second_head = last_layer_attention[1].cpu().numpy()
sns.heatmap(second_head, annot=True, fmt='.2f', cmap='Blues', ax=axes[1], 
            xticklabels=tokens, yticklabels=tokens, cbar_kws={'label': 'Attention Weight'})
axes[1].set_title('BERT - Last Layer, Head 2', fontsize=13)
axes[1].set_xlabel('Attending to (Keys)', fontsize=11)
axes[1].set_ylabel('Attending from (Queries)', fontsize=11)

plt.tight_layout()
plt.show()

print('\nKey observations:')
print('- [CLS] and [SEP] tokens attend to all tokens')
print('- Related tokens (e.g., "cat", "sat") show strong attention')
print('- Different heads capture different patterns (syntactic vs semantic)')

---
## 7. Masked Attention

**Causal Mask (Decoder):**
- Prevent tokens from attending to future positions
- Ensures autoregressive property

**Padding Mask:**
- Exclude padding tokens from attention
- Important for variable-length sequences

In [ ]:
def create_causal_mask(seq_len):
    """
    Create lower-triangular causal mask.
    Position i can only attend to positions <= i.
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

def create_padding_mask(lengths, max_len):
    """
    Create padding mask for variable-length sequences.
    Real tokens: 1, Padding tokens: 0
    """
    batch_size = len(lengths)
    mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)
    return mask.float()

# Demonstrate masks
seq_len = 5

causal_mask = create_causal_mask(seq_len)
padding_mask = create_padding_mask(torch.tensor([5, 3, 4]), max_len=5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Causal mask
sns.heatmap(causal_mask, annot=True, fmt='.0f', cmap='RdYlGn', ax=axes[0], 
            cbar_kws={'label': 'Can Attend'})
axes[0].set_title('Causal Mask (Decoder)', fontsize=12)
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')

# Padding mask
sns.heatmap(padding_mask, annot=True, fmt='.0f', cmap='RdYlGn', ax=axes[1],
            cbar_kws={'label': 'Not Padding'})
axes[1].set_title('Padding Mask', fontsize=12)
axes[1].set_xlabel('Token Position')
axes[1].set_ylabel('Batch Sample')

# Combined: causal + padding
combined_mask = causal_mask.unsqueeze(0) * padding_mask.unsqueeze(-1)
sns.heatmap(combined_mask[0], annot=True, fmt='.0f', cmap='RdYlGn', ax=axes[2],
            cbar_kws={'label': 'Can Attend'})
axes[2].set_title('Combined Mask (Causal + Padding)', fontsize=12)
axes[2].set_xlabel('Key Position')
axes[2].set_ylabel('Query Position')

plt.tight_layout()
plt.show()

# Example: Apply mask to attention scores
batch_size, seq_len, d_k = 2, 4, 8
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

# Compute scores
scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
print(f'\nScores before masking:\n{scores[0]}')

# Apply causal mask
causal_mask = create_causal_mask(seq_len).unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)
scores_masked = scores.masked_fill(causal_mask == 0, float('-inf'))
print(f'\nScores after causal mask (inf = masked):\n{scores_masked[0]}')

# Softmax (inf becomes 0)
attn_weights = F.softmax(scores_masked, dim=-1)
print(f'\nAttention weights after softmax:\n{attn_weights[0]}')

---
## 8. Relative Positional Encoding

**Problem:** Absolute positional encodings don't capture relative distances well.

**ALiBi (Attention with Linear Biases):**
- Add learned biases directly to attention scores
- Simpler, extrapolates better to longer sequences

**RoPE (Rotary Position Embeddings):**
- Rotate Q and K by angle dependent on position
- Encodes position in the rotation angle
- Excellent extrapolation properties

In [ ]:
def apply_alibi(scores, num_heads):
    """
    Apply ALiBi (Attention with Linear Biases) to attention scores.
    Adds linear biases based on relative position.
    """
    seq_len = scores.shape[-1]
    # Create distance matrix
    distances = torch.arange(seq_len).unsqueeze(1) - torch.arange(seq_len).unsqueeze(0)
    
    # Different slopes for different heads
    slopes = torch.arange(1, num_heads + 1, dtype=torch.float32)
    slopes = 1.0 / (2 ** slopes)  # Geometric progression
    
    # Apply slopes
    alibi_bias = distances.unsqueeze(0) * slopes.view(-1, 1, 1)
    
    return scores + alibi_bias

def rope_embedding(x, d_k, position_ids=None):
    """
    Apply RoPE (Rotary Position Embeddings).
    Rotates Q and K by angles dependent on position.
    """
    if position_ids is None:
        position_ids = torch.arange(x.shape[-2], dtype=torch.float32, device=x.device)
    
    # Create rotation angles
    dim_indices = torch.arange(0, d_k, 2, dtype=torch.float32, device=x.device)
    inv_freq = 1.0 / (10000 ** (dim_indices / d_k))
    
    # Position * frequency
    t = position_ids.unsqueeze(-1) * inv_freq.unsqueeze(0)  # (seq_len, d_k/2)
    
    # Compute cos and sin
    cos_cached = torch.cos(t)  # (seq_len, d_k/2)
    sin_cached = torch.sin(t)
    
    # Apply rotation to x: [x1, x2, x3, x4] -> [x1*cos-x2*sin, x1*sin+x2*cos, x3*cos-x4*sin, ...]
    x1 = x[..., 0::2] * cos_cached - x[..., 1::2] * sin_cached
    x2 = x[..., 0::2] * sin_cached + x[..., 1::2] * cos_cached
    
    # Interleave
    x_rotated = torch.zeros_like(x)
    x_rotated[..., 0::2] = x1
    x_rotated[..., 1::2] = x2
    
    return x_rotated

# Visualization: Position encoding decay
seq_len = 20
num_heads = 8

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ALiBi biases
distances = torch.arange(seq_len).unsqueeze(1) - torch.arange(seq_len).unsqueeze(0)
for head in range(4):
    slope = 1.0 / (2 ** (head + 1))
    bias = distances[0, :] * slope  # Distance from first token
    axes[0].plot(bias.numpy(), marker='o', label=f'Head {head+1}', linewidth=2)

axes[0].set_xlabel('Distance from Position 0', fontsize=12)
axes[0].set_ylabel('ALiBi Bias', fontsize=12)
axes[0].set_title('ALiBi: Linear Bias per Head', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RoPE: Show how angles vary with position
d_k = 16
positions = torch.arange(seq_len, dtype=torch.float32)
dim_indices = torch.arange(0, d_k, 2)
inv_freq = 1.0 / (10000 ** (dim_indices / d_k))

angles = positions.unsqueeze(-1) * inv_freq.unsqueeze(0)  # (seq_len, d_k/2)
for dim in range(4):
    axes[1].plot(angles[:, dim].numpy(), marker='o', label=f'Dim {dim}', linewidth=2)

axes[1].set_xlabel('Position', fontsize=12)
axes[1].set_ylabel('Rotation Angle (radians)', fontsize=12)
axes[1].set_title('RoPE: Rotation Angles by Position', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Position Encoding Comparison:')
print('\n1. Absolute Positional Encoding:')
print('   - Add sine/cosine embeddings to token embeddings')
print('   - Doesn\'t extrapolate well beyond training sequence length')
print('\n2. ALiBi (Attention with Linear Biases):')
print('   - No learned parameters, just add linear biases')
print('   - Simpler, better extrapolation')
print('\n3. RoPE (Rotary Position Embeddings):')
print('   - Encodes position in rotation angles')
print('   - Excellent extrapolation, used in modern models (LLaMA, Falcon)')

---
## Interview Takeaways

### Core Concepts
1. **Scaled Dot-Product Attention**: Foundation of all modern attention. Scaling by $\sqrt{d_k}$ prevents softmax saturation.

2. **Bahdanau vs Luong**:
   - Bahdanau (additive): More flexible, slower. $score = v^T \tanh(W_1 Q + W_2 K)$
   - Luong (multiplicative): Faster, simpler. $score = Q^T W K$. Used in Transformers.

3. **Multi-Head Attention**:
   - h parallel attention heads with different learned projections
   - Each head specializes in different types of relationships
   - Concatenate and project outputs

4. **Self vs Cross-Attention**:
   - **Self**: Q, K, V from same sequence → tokens attend to each other
   - **Cross**: Q from target, K,V from source → enables encoder-decoder alignment

5. **KV-Cache**:
   - Store K,V from previous steps during generation
   - Reduces complexity from O(n²) to O(n) for decoding
   - ~100x speedup for long sequences

6. **Masking**:
   - **Causal**: Prevent attending to future (decoder)
   - **Padding**: Exclude padding tokens from attention

7. **Position Encoding**:
   - **Absolute**: Sin/cos, doesn't generalize to longer sequences
   - **ALiBi**: Linear biases, simpler, better extrapolation
   - **RoPE**: Rotary embeddings, state-of-the-art extrapolation

### Key Formulas
$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1,...,\text{head}_h)W^O$$

### Common Interview Questions
- **Why scale by $\sqrt{d_k}$?** → Prevents attention weights from becoming too peaked (gradients vanish)
- **Why use attention instead of RNNs?** → Parallel computation, longer context, no vanishing gradients
- **What's the difference between Transformers and RNNs?** → Transformers use attention (parallel), RNNs are sequential
- **How does KV-Cache work in practice?** → Prefill (compute all K,V), then decode (reuse cached K,V for new tokens)
- **Why do you need masking?** → Prevent information leakage (causal), handle variable lengths (padding)
- **Computational complexity?** → Self-attention: O(n²d), K,V-cache reduces decode to O(nd)

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>